# 🧠 05 · Deep learning in one afternoon (PyTorch)
### From a straight line to a 3D convolutional network that classifies noisy molecules — and the traps (data leakage, bad augmentation) that fool even experts

**BiologyToResearch** · Part A — Foundations · Module 05 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/05_deep_learning_pytorch.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU is enough (≈ 3–5 min to run everything); a T4 GPU is used automatically if present |
| **Core frameworks** | PyTorch (tensors, autograd, `nn`, `DataLoader`), NumPy, SciPy, scikit-learn (metrics), Matplotlib |
| **Prerequisites** | Module 04 (noise, SNR, precision/recall); Modules 01–03 (arrays, rotations, missing wedge basics) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Preview of Part D: the SHREC'19/'20 cryo-ET classification benchmarks (Gubins *et al.*, with Xu M); the lab's first deep subtomogram classifier (Xu *et al.* 2017); DISCA's augmentation choices (Zeng *et al.* 2023) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments in this course are small, controlled *toy versions* of published methods. They are designed to teach the
> core idea of each paper; they do **not** reproduce the papers' numbers. Networks here are deliberately tiny and
> trained for seconds on simulated data. **Nothing is stored in the repository:** molecules are real Protein Data Bank
> (PDB) models, downloaded the first time a notebook needs them and cached on your computer (offline, code-built
> stand-ins of the same size and symmetry are used instead); all images and data are simulated by code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Use** PyTorch tensors and automatic differentiation to fit a model by gradient descent.
2. **Explain** neurons, activation functions, multilayer perceptrons (MLPs), softmax and the cross-entropy loss, and
   **write** a complete training loop with `Dataset`/`DataLoader`.
3. **Explain** what a convolution layer learns and **build** 2D and 3D convolutional neural networks (CNNs).
4. **Demonstrate** data leakage: show that splitting particles (instead of tomograms) between training and test sets
   produces falsely optimistic accuracy.
5. **Measure** overfitting and **compare** augmentation strategies — and explain why a careless rotation augmentation
   can *hurt*.
6. **Train and evaluate** a 2D CNN on noisy projections and a 3D CNN on 24³ subtomograms with a missing wedge, using
   confusion matrices and per-class precision/recall.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, device, toolkit, a bank of molecule projections | 8 min |
| 1 · Tensors and autograd | fit a straight line by gradient descent | 15 min |
| 2 · Neurons, activations and an MLP | "inside or outside the vesicle?" | 15 min |
| 3 · The training recipe | softmax, cross-entropy, `Dataset`, `DataLoader`, a reusable `fit` | 15 min |
| 4 · Convolutional networks | learned filters = learned templates | 15 min |
| 5 · Splits and data leakage | particle-level vs tomogram-level splitting | 15 min |
| 6 · Overfitting and augmentation | few labels; good and bad rotation augmentation | 15 min |
| 7 · A 2D CNN for four molecules | train, validate, confusion matrix, robustness | 20 min |
| 8 · A tiny 3D CNN on subtomograms | missing wedge, wedge-preserving augmentation | 20 min |
| 9 · GPUs, benchmarks and what comes next | SHREC, the lab's first deep classifier | 7 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A single cellular tomogram contains thousands of particles; a study may have hundreds of tomograms and **millions of
subtomograms**. Template matching (Module 11) needs a known structure and is slow; classical classification (Module 14)
compares every particle with every other. Since 2017, most methods of the Xu Lab are **neural networks** that learn,
from examples, to recognise molecules in noisy 3D images in milliseconds per particle — or, without labels at all, to
sort them (Modules 16–32).

Neural networks are not magic. They are functions with many adjustable numbers, tuned by following the slope of an
error. In this notebook you will build one from scratch, train it to tell **ribosomes, proteasomes, capped proteasomes
and RuBisCO** apart in noisy images — and learn the two mistakes that make published accuracies meaningless:
**leaky data splits** and **augmentations that do not match the physics**.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, every network here is small enough for a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import copy
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

The helpers below are copied verbatim from [`course_toolkit.py`](../course_toolkit.py): building and preparing zoo
maps, random rotations, the missing-wedge mask, noise at a chosen SNR, and `simulate_subtomogram`, which makes one
randomly rotated, wedge-filtered, noisy subtomogram (Modules 02–04, 07).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def simulate_subtomogram(template: np.ndarray, rng: np.random.Generator, snr: float = 0.1,
                         tilt_max_deg: float | None = 60.0, max_shift: float = 0.0,
                         R: np.ndarray | None = None):
    """Randomly rotate/shift a template, impose a missing wedge and add noise.

    Returns (subtomogram, R, shift_xyz).  This 'direct' simulation (Fourier mask +
    white noise) is the fast approximation used by many Xu-lab benchmarks.
    """
    R = random_rotation(rng) if R is None else R
    t = rng.uniform(-max_shift, max_shift, 3) if max_shift > 0 else np.zeros(3)
    v = rotate_volume(template, R, t)
    if tilt_max_deg is not None:
        v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng), R, t
print("🧰 toolkit loaded")

### Data: a bank of clean projections of four molecules

We build the four "star" molecules (10 Å voxels, 32³ boxes) and make **700 clean 2D projections per molecule**, each
from a uniformly random 3D orientation — what a microscope would record of particles lying randomly in ice (Module 06).
Noise is added later, differently in each experiment. Making the bank takes a few seconds.

In [ ]:
# ── Load the zoo and build a bank of clean projections ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
CLASS_NAMES = ["ribosome", "20S proteasome", "capped proteasome", "RuBisCO"]
zoo = load_zoo(CAST, size=32)
N_PER_CLASS = 700
t0 = time.time()
bank = np.zeros((len(CAST), N_PER_CLASS, 32, 32), np.float32)      # [class, copy, y, x]
for c, pid in enumerate(CAST):
    for j in range(N_PER_CLASS):
        bank[c, j] = rotate_volume(zoo[pid], random_rotation(rng)).sum(axis=0)   # projection along z (the beam)
    bank[c] /= bank[c].max()
print(f"bank of {bank.shape[0]} × {bank.shape[1]} clean 32×32 projections (1 nm pixels) in {time.time() - t0:.1f} s")

fig, axes = plt.subplots(len(CAST), 6, figsize=(7.5, 5.2))
for c in range(len(CAST)):
    for j in range(6):
        axes[c, j].imshow(bank[c, j]); axes[c, j].set_xticks([]); axes[c, j].set_yticks([])
    axes[c, 0].set_ylabel(CLASS_NAMES[c], fontsize=8)
fig.suptitle("six random views of each molecule (noise-free)", fontsize=10)
plt.tight_layout(); plt.show()

💡 Notice how different the *same* molecule looks from different directions — and how similar the 20S proteasome and
the capped proteasome are (they differ only by a cap at one end). A classifier must learn what stays the same across
views. That is the whole challenge.

---
## 1 · Tensors and automatic differentiation

> 📍 **Where we are.** Module 04 fitted models by formulas (EM, maximum likelihood) and scored them (accuracy, F1).
> Neural networks are models too complex for formulas: we fit them by repeatedly nudging their parameters downhill on
> an error surface. This module builds that machinery and uses it on our zoo; Part D (Modules 16–20) scales it up.

A **tensor** is PyTorch's version of a NumPy array — same idea (a grid of numbers with a shape), plus two superpowers:
it can live on a **GPU**, and it can **remember how it was computed**, so PyTorch can work out derivatives automatically
(**autograd**).

🧫 **A biology example.** Faster-growing bacteria devote a larger fraction of their protein to ribosomes — an almost
linear "growth law" (Scott *et al.* 2010). We generate **synthetic** measurements of ribosome fraction $y$ vs growth rate
$x$ and fit a line $\hat y = w x + b$ by minimising the **mean squared error (MSE)**:

$$L(w,b) = \frac1n\sum_{i=1}^{n}\big(w x_i + b - y_i\big)^2 .$$

**Gradient descent** repeats one rule: compute the slopes $\partial L/\partial w$ and $\partial L/\partial b$ (the
*gradient*: how much the error changes if we nudge each parameter), then step a little *downhill*:

$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b}.$$

*In words:* the **learning rate** $\eta$ sets the step size; too small is slow, too large overshoots. You never write
the derivatives yourself — `loss.backward()` computes them.

In [ ]:
# ── 1.1 Tensors: like NumPy arrays, plus gradients ──
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("tensor:", a, "\nshape:", tuple(a.shape), "| dtype:", a.dtype, "| device:", a.device)
print("from NumPy and back:", torch.from_numpy(np.arange(3.0)), "→", torch.arange(3.0).numpy())
w = torch.tensor(3.0, requires_grad=True)          # "track operations on w"
y = w ** 2 + 2 * w                                 # y = w² + 2w
y.backward()                                       # dy/dw = 2w + 2 = 8 at w = 3
print("autograd: d(w² + 2w)/dw at w = 3 →", w.grad.item())

In [ ]:
# ── 1.2 Fit the growth law by gradient descent ──
growth = torch.tensor(rng.uniform(0.2, 2.0, 40), dtype=torch.float32)                       # doublings per hour
ribo_frac = 0.07 + 0.18 * growth + torch.tensor(rng.normal(0, 0.02, 40), dtype=torch.float32)  # synthetic data

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr, history = 0.1, []
for step in range(300):
    loss = ((w * growth + b - ribo_frac) ** 2).mean()   # forward pass: compute the error
    loss.backward()                                     # backward pass: fill w.grad and b.grad
    with torch.no_grad():                               # update without recording these operations
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()                      # gradients accumulate, so reset them
    history.append(loss.item())
print(f"gradient descent: slope w = {w.item():.3f}, intercept b = {b.item():.3f}  (data made with 0.18 and 0.07)")
coef = np.polyfit(growth.numpy(), ribo_frac.numpy(), 1)
print(f"least-squares formula (np.polyfit): slope {coef[0]:.3f}, intercept {coef[1]:.3f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
a1.scatter(growth, ribo_frac, s=12); g = torch.linspace(0.2, 2.0, 50)
a1.plot(g, (w * g + b).detach(), "tab:red", label="fitted line")
a1.set_xlabel("growth rate (doublings / h)"); a1.set_ylabel("ribosomal protein fraction"); a1.legend(fontsize=7)
a1.set_title("synthetic 'growth law' data")
a2.plot(history); a2.set_yscale("log"); a2.set_xlabel("gradient-descent step"); a2.set_ylabel("MSE loss")
a2.set_title("the loss goes downhill"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Starting from $w=b=0$, 300 small downhill steps found the same line as the exact
least-squares formula. For a straight line we did not need gradient descent — but for a network with a million
parameters there is no formula, and exactly this loop (forward → loss → `backward()` → step) is how it is trained.

✏️ **Your turn.** Set `lr = 1.2` and re-run. What happens to the loss? Then try `lr = 0.01`. This is the most important
"knob" in deep learning.

---
## 2 · Neurons, activation functions and a multilayer perceptron

An artificial **neuron** computes a weighted sum of its inputs plus a bias, then passes it through a nonlinear
**activation function** $\phi$:

$$h = \phi\Big(\sum_j w_j x_j + b\Big).$$

A **layer** is many neurons reading the same inputs; a **multilayer perceptron (MLP)** stacks layers, each feeding the
next. Without the nonlinearity $\phi$, any stack of layers would collapse into a single linear function (a line or a
plane); with it, MLPs can draw curved decision boundaries. The most common activation today is the **ReLU**,
$\phi(z)=\max(0,z)$ — "pass positive signals, block negative ones".

🧫 **Toy problem: inside or outside the vesicle?** Particles at positions $(x, y)$ (nm) near a membrane vesicle of radius
100 nm; the label says whether a particle is inside. No straight line can separate inside from outside a circle.

In [ ]:
# ── 2.1 Activation functions ──
z = torch.linspace(-4, 4, 200)
fig, ax = plt.subplots(figsize=(5, 2.8))
for f, name in [(torch.relu, "ReLU  max(0, z)"), (torch.sigmoid, "sigmoid  1/(1+e^{-z})"), (torch.tanh, "tanh")]:
    ax.plot(z, f(z), label=name)
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("weighted input z"); ax.set_ylabel("output φ(z)")
ax.set_title("activation functions add the nonlinearity"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 A linear model vs an MLP on the vesicle problem ──
pos = torch.tensor(rng.uniform(-200, 200, (1000, 2)), dtype=torch.float32)     # positions in nm
inside = (pos.norm(dim=1) < 100).long()                                       # 1 = inside the vesicle
x_in = pos / 100.0                                                            # rescale inputs to ~[-2, 2]

linear = nn.Linear(2, 2)                                                      # logistic regression (2 classes)
mlp = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 2))
for model in (linear, mlp):
    opt = torch.optim.Adam(model.parameters(), lr=0.03)
    for step in range(400):
        loss = F.cross_entropy(model(x_in), inside)                          # explained in Section 3
        opt.zero_grad(); loss.backward(); opt.step()
n_params = sum(p.numel() for p in mlp.parameters())
print(f"MLP has {n_params} adjustable numbers (weights and biases)")

gx, gy = torch.meshgrid(torch.linspace(-2, 2, 150), torch.linspace(-2, 2, 150), indexing="xy")
grid_pts = torch.stack([gx.ravel(), gy.ravel()], 1)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
for ax, model, name in zip(axes, (linear, mlp), ("linear model (one layer)", "MLP (2 hidden layers of 16 ReLUs)")):
    with torch.no_grad():
        acc = (model(x_in).argmax(1) == inside).float().mean().item()
        region = model(grid_pts).argmax(1).reshape(150, 150)
    ax.contourf(gx * 100, gy * 100, region, levels=[-0.5, 0.5, 1.5], colors=["#f3e3d3", "#d3e3f3"])
    ax.scatter(pos[:, 0], pos[:, 1], c=inside, cmap="coolwarm", s=4)
    ax.add_patch(plt.Circle((0, 0), 100, fill=False, color="k", lw=1.5))
    ax.set_aspect("equal"); ax.set_title(f"{name}\naccuracy {acc:.2f}", fontsize=9); ax.set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The linear model can only cut the plane with a straight line, so it does little better than
guessing "outside" for everything. The MLP — a few hundred parameters, ReLUs in between — bends its decision boundary
into a near-circle around the vesicle. Every network in this course is built from such layers; convolutional layers
(Section 4) are a special kind designed for images.

---
## 3 · The training recipe: logits, softmax, cross-entropy, data loaders

### 3.1 From scores to probabilities to a loss

A classifier with $K$ classes outputs $K$ raw scores called **logits** $z_1,\dots,z_K$. The **softmax** turns them into
probabilities, and the **cross-entropy** loss is the negative log-probability given to the *correct* class $y$:

$$p_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}, \qquad L_{\mathrm{CE}} = -\log p_y .$$

*In words:* softmax makes all scores positive and sum to 1; cross-entropy is 0 if the network gives the right class
probability 1, and grows without bound as that probability approaches 0. It is the classification counterpart of the
MSE (and it *is* a negative log-likelihood — Module 04, Section 6).

In [ ]:
# ── 3.1 Softmax and cross-entropy by hand vs PyTorch ──
logits = torch.tensor([[2.0, 0.5, -1.0, 0.1]])        # network scores for ribosome, 20S, capped, RuBisCO
true_class = torch.tensor([0])                         # it really is a ribosome
p = torch.exp(logits) / torch.exp(logits).sum()
print("softmax probabilities:", np.round(p.numpy().ravel(), 3), "(sum =", round(p.sum().item(), 3), ")")
print(f"cross-entropy by hand: {-torch.log(p[0, 0]).item():.4f}   F.cross_entropy: {F.cross_entropy(logits, true_class).item():.4f}")
print(f"if the true class were RuBisCO instead: {F.cross_entropy(logits, torch.tensor([3])).item():.4f}  (a confident mistake costs more)")

### 3.2 Datasets, mini-batches and epochs

Real training sets do not fit in one gradient step. We shuffle the examples and feed them in **mini-batches** (e.g. 64
images); one pass over all training data is an **epoch**. PyTorch splits the job in two:

- a **`Dataset`** knows how to return example *i* (and can apply random **augmentation** on the fly);
- a **`DataLoader`** shuffles, groups examples into batches and hands them to the training loop.

We also write one reusable `fit` function — the training loop you will reuse in every later module — that records the
**training** loss and accuracy, evaluates on a separate **validation** set, and keeps the weights from the best
validation epoch (a simple form of **early stopping**). Updates use **Adam**, a popular variant of gradient descent that
adapts the step size of each parameter.

In [ ]:
# ── 3.2 A Dataset, an evaluation helper and a reusable training loop ──
class ParticleDataset(Dataset):
    """Images (or volumes) with integer labels; optional random augmentation applied each time an item is read."""

    def __init__(self, images: np.ndarray, labels: np.ndarray, augment=None):
        self.x = torch.as_tensor(images, dtype=torch.float32).unsqueeze(1)   # add a channel axis: [n, 1, ...]
        self.y = torch.as_tensor(labels, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.x[i]
        return (self.augment(x) if self.augment else x), self.y[i]


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    """Return (mean cross-entropy, accuracy, predicted labels, true labels) of a model on a DataLoader."""
    model.eval()
    losses, preds, trues = [], [], []
    for xb, yb in loader:
        out = model(xb.to(DEVICE))
        losses.append(F.cross_entropy(out, yb.to(DEVICE), reduction="sum").item())
        preds.append(out.argmax(1).cpu()); trues.append(yb)
    preds, trues = torch.cat(preds).numpy(), torch.cat(trues).numpy()
    return sum(losses) / len(trues), float((preds == trues).mean()), preds, trues


def fit(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader | None = None, epochs: int = 10,
        lr: float = 1e-3, weight_decay: float = 0.0, eval_every: int = 1, verbose: bool = True) -> dict:
    """Train with Adam + cross-entropy; keep the weights of the best validation epoch. Returns a history dict."""
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state, t0 = -1.0, None, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            loss = F.cross_entropy(model(xb.to(DEVICE)), yb.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % eval_every == 0 or ep == epochs:
            tl, ta, _, _ = evaluate(model, train_loader)
            vl, va, _, _ = evaluate(model, val_loader) if val_loader is not None else (np.nan, np.nan, None, None)
            for k, v in zip(hist, (ep, tl, ta, vl, va)):
                hist[k].append(v)
            if val_loader is not None and va > best_acc:
                best_acc, best_state = va, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"  epoch {ep:3d}: train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")
    if best_state is not None:
        model.load_state_dict(best_state)          # early stopping: restore the best validation epoch
    hist["seconds"] = time.time() - t0
    if verbose:
        print(f"  trained in {hist['seconds']:.1f} s on {DEVICE}")
    return hist


print("ParticleDataset, evaluate() and fit() are ready")

---
## 4 · Convolutional neural networks: learned templates

An MLP treats an image as a list of unrelated numbers. A **convolution layer** instead slides small **filters**
(kernels, e.g. 3 × 3 pixels) over the image and, at every position, computes a weighted sum — exactly the
cross-correlation of Module 03, but with filters that are **learned**:

$$\text{out}_f(y, x) = b_f + \sum_{c}\sum_{i,j} w_{f,c,i,j}\;\text{in}_c(y+i,\, x+j).$$

*In words:* each filter $f$ is a tiny template; its output **feature map** lights up wherever the input locally matches
it. Because the same filter is used everywhere, a CNN recognises a pattern wherever it appears and has far fewer
parameters than an MLP. **Max-pooling** (keep the largest value in each 2 × 2 block) halves the size, so deeper layers see
larger regions (a larger **receptive field**) — edges → blobs → subunits → whole molecules. A final fully connected
"head" turns the feature maps into class logits.

Our network (`SmallCNN2D`): two convolution + ReLU + max-pool blocks (32 → 16 → 8 pixels), then an MLP head. The Xu-lab
networks of Module 16 (e.g. DSRF3D) follow the same plan with more layers.

In [ ]:
# ── 4.1 A small 2D CNN ──
class SmallCNN2D(nn.Module):
    """conv(8) → ReLU → maxpool → conv(16) → ReLU → maxpool → flatten → FC(64) → ReLU → FC(n_classes)."""

    def __init__(self, n_classes: int = 4, size: int = 32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * (size // 4) ** 2, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


demo = SmallCNN2D()
x_demo = torch.zeros(5, 1, 32, 32)                        # a batch of 5 one-channel 32×32 images
print("input", tuple(x_demo.shape), "→ features", tuple(demo.features(x_demo).shape), "→ logits", tuple(demo(x_demo).shape))
for name, p in demo.named_parameters():
    print(f"  {name:18s} {str(tuple(p.shape)):16s} {p.numel():6d} parameters")
print("total:", sum(p.numel() for p in demo.parameters()), "parameters")

🔍 **What just happened?** The shapes trace the data through the network: 1 channel × 32 × 32 → 16 feature maps of
8 × 8 → 4 logits. The first convolution has only 8 × 9 weights + 8 biases = 80 parameters; most parameters sit in the
fully connected head. Images enter as `[batch, channel, y, x]`, volumes as `[batch, channel, z, y, x]`.

---
## 5 · Train, validation, test — and why the split *unit* matters

A model's accuracy on data it was trained on says little: it may have memorised them. We therefore split the data:

| Set | Used for | Rule |
|---|---|---|
| **training** | adjusting the weights | the network sees it every epoch |
| **validation** | choosing settings (epochs, learning rate, architecture) | never used for gradient steps |
| **test** | the final, reported number | touched **once**, at the very end |

⚠️ **Data leakage** happens when information from the test set sneaks into training, making results look better than
they will be on new data. In cryo-ET the classic leak is the **split unit**. Particles from the same tomogram share
its ice thickness, defocus, contrast, background gradients and often its composition (a ribosome-rich region, a
membrane-rich lamella). If you split *particles* at random, the test set contains siblings of training particles from the
same tomograms — and a network can score well by recognising **the tomogram**, not the molecule. The honest protocol is
to split **by tomogram** (or by cell, by grid, by day).

🧪 **Experiment.** We simulate 16 "tomograms" of 80 particles. Each tomogram has (i) a fingerprint — its own smooth
background pattern and contrast — and (ii) a biased composition (70 % of its particles are one "dominant" species).
Particle images are noisy (SNR 0.2). We train the same MLP twice: with a random **particle-level** split and with a
**tomogram-level** split (4 whole tomograms held out, one per dominant species).

In [ ]:
# ── 5.1 Simulate 16 tomograms with fingerprints and biased compositions ──
rng = np.random.default_rng(SEED + 5)                   # re-seed so this experiment does not depend on earlier cells


def smooth_field(rng, shape=(32, 32), sigma: float = 4.0) -> np.ndarray:
    """A random smooth 2D pattern with unit standard deviation (stands in for uneven ice / background)."""
    f = ndimage.gaussian_filter(rng.normal(size=shape), sigma)
    return f / f.std()


def standardize(img: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation per image (the usual input normalisation)."""
    return ((img - img.mean()) / (img.std() + 1e-8)).astype(np.float32)


N_TOMO, PER_TOMO = 16, 80
dominant = np.repeat(np.arange(4), 4)                   # tomograms 0–3 ribosome-rich, 4–7 20S-rich, ...
unused = {c: list(rng.permutation(N_PER_CLASS)) for c in range(4)}   # draw each clean projection at most once
leak_x, leak_y, leak_tomo = [], [], []
for t in range(N_TOMO):
    fingerprint, gain = 0.5 * smooth_field(rng) * bank.std(), rng.uniform(0.7, 1.3)
    for _ in range(PER_TOMO):
        c = dominant[t] if rng.random() < 0.7 else int(rng.integers(4))
        img = gain * bank[c, unused[c].pop()] + fingerprint
        leak_x.append(standardize(add_noise_snr(img, 0.2, rng))); leak_y.append(c); leak_tomo.append(t)
leak_x, leak_y, leak_tomo = np.array(leak_x), np.array(leak_y), np.array(leak_tomo)
print(f"{len(leak_y)} particles in {N_TOMO} tomograms; class counts {np.bincount(leak_y)}")

fig, axes = plt.subplots(2, 6, figsize=(9, 3.9))
for j, t in enumerate([0, 5, 10]):
    idx = np.where(leak_tomo == t)[0][:4]
    for k, i in enumerate(idx):
        ax = axes[k // 2, 2 * j + k % 2]
        ax.imshow(leak_x[i]); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"tomo {t}: {CLASS_NAMES[leak_y[i]].split()[0]}", fontsize=7)
fig.suptitle("particles from three tomograms — each tomogram has its own faint background 'fingerprint'", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.2 Particle-level vs tomogram-level split (same model, same data) ──
def train_mlp_and_test(train_idx: np.ndarray, test_idx: np.ndarray, epochs: int = 30) -> tuple[float, float]:
    """Train a small MLP on leak_x[train_idx]; return (training accuracy, test accuracy)."""
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Flatten(), nn.Linear(32 * 32, 64), nn.ReLU(), nn.Linear(64, 4))
    train_loader = DataLoader(ParticleDataset(leak_x[train_idx], leak_y[train_idx]), batch_size=64, shuffle=True)
    test_loader = DataLoader(ParticleDataset(leak_x[test_idx], leak_y[test_idx]), batch_size=256)
    fit(model, train_loader, None, epochs=epochs, lr=1e-3, weight_decay=1e-4, eval_every=epochs, verbose=False)
    return evaluate(model, train_loader)[1], evaluate(model, test_loader)[1]


t0 = time.time()
perm = rng.permutation(len(leak_y))
split_particles = (perm[:960], perm[960:])                                    # 75 % / 25 % of particles
test_tomos = [0, 4, 8, 12]                                                    # one tomogram per dominant species
split_tomograms = (np.where(~np.isin(leak_tomo, test_tomos))[0], np.where(np.isin(leak_tomo, test_tomos))[0])
acc_p = train_mlp_and_test(*split_particles)
acc_t = train_mlp_and_test(*split_tomograms)
print(f"particle-level split : train acc {acc_p[0]:.2f}, test acc {acc_p[1]:.2f}")
print(f"tomogram-level split : train acc {acc_t[0]:.2f}, test acc {acc_t[1]:.2f}   ({time.time() - t0:.1f} s)")
assert acc_p[1] > acc_t[1], "the leaky split should look better than the honest one"

fig, ax = plt.subplots(figsize=(4.6, 2.8))
ax.bar(["particle-level\n(leaky)", "tomogram-level\n(honest)"], [acc_p[1], acc_t[1]], color=["tab:red", "tab:green"])
ax.axhline(0.25, color="k", ls="--", lw=1, label="chance (4 classes)")
ax.set_ylabel("test accuracy"); ax.set_ylim(0, 1); ax.legend(fontsize=7)
ax.set_title("same network, same data, different split"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Both models fit their training data perfectly. The particle-level split *reports* a clearly
higher test accuracy — because its test particles come from tomograms the network has already seen, and it learned
shortcuts: "this background pattern means a ribosome-rich tomogram". On **unseen tomograms**, the honest number is
lower. A reviewer cannot detect this from the paper's accuracy alone; only the description of the split reveals it.

Other common leaks in our field: augmenting (e.g. rotating) particles **before** splitting, so rotated copies of one
particle land in both sets; overlapping picks of the same particle; choosing hyper-parameters on the test set; and
normalising with statistics computed on all data. The SHREC benchmarks (Section 9) avoid the main one by holding out a
whole tomogram for testing — the same pseudo-replication lesson as Module 04's membranes.

✏️ **Your turn.** Set the fingerprint amplitude `0.5` to `0.0` in cell 5.1 and re-run 5.1–5.2. Does the gap close?

---
## 6 · Overfitting and data augmentation

With few labelled examples, a network with thousands of parameters can memorise every training image — including its
noise — and fail on new ones: **overfitting**. Two classic remedies are more data and **data augmentation**: showing the
network transformed copies of each training example that preserve the label.

🧫 For molecules, the natural augmentation is **rotation** — a ribosome is a ribosome in any orientation. But an
augmentation must also **preserve the image statistics**. Rotating a noisy pixel image by an arbitrary angle requires
interpolation, which (i) smooths the noise (averaging neighbouring pixels lowers its variance) and (ii) leaves empty,
noise-free corners. The network then trains on images unlike any real test image. Rotations by **multiples of 90°** and
**flips** move pixels exactly, without interpolation. For a *projection*, a flip is also physically legitimate: it is the
projection of the molecule rotated by 180° about an in-plane axis.

🧪 **Experiment.** Only **10 labelled images per class** (SNR 0.2), tested on 800 unseen images. We compare: no
augmentation; arbitrary-angle rotation with interpolation; exact 90° rotations + flips.

In [ ]:
# ── 6.1 Two rotation augmentations ──
def augment_interpolated_rotation(x: torch.Tensor) -> torch.Tensor:
    """Rotate one [1, H, W] image by a random angle with bilinear interpolation (zero-filled corners)."""
    a = float(torch.rand(1)) * 2 * np.pi
    theta = torch.tensor([[[np.cos(a), -np.sin(a), 0.0], [np.sin(a), np.cos(a), 0.0]]], dtype=torch.float32)
    grid = F.affine_grid(theta, (1, *x.shape), align_corners=False)
    return F.grid_sample(x[None], grid, align_corners=False, padding_mode="zeros")[0]


def augment_rot90_flip(x: torch.Tensor) -> torch.Tensor:
    """Exact symmetry of the pixel grid: rotate by a random multiple of 90° and flip with probability 1/2."""
    x = torch.rot90(x, int(torch.randint(4, (1,))), dims=(-2, -1))
    return x.flip(-1) if torch.rand(1) < 0.5 else x


rng = np.random.default_rng(SEED + 6)
noisy_bank = np.stack([[standardize(add_noise_snr(bank[c, j], 0.2, rng)) for j in range(N_PER_CLASS)] for c in range(4)])
example = torch.tensor(noisy_bank[0, 0])[None]
fig, axes = plt.subplots(1, 4, figsize=(8, 2.3))
for ax, (img, t) in zip(axes, [(example, "original (noisy)"), (augment_interpolated_rotation(example), "interpolated rotation"),
                               (augment_rot90_flip(example), "rot90 + flip"), (augment_rot90_flip(example), "rot90 + flip")]):
    ax.imshow(img[0]); ax.axis("off"); ax.set_title(f"{t}\npixel std {img.std():.2f}", fontsize=8)
plt.tight_layout(); plt.show()

Look at the pixel standard deviations: interpolation makes the noise *smoother* (lower std) and the corners empty.
Now the experiment.

In [ ]:
# ── 6.2 Few labels: no augmentation vs two rotation augmentations ──
few_idx = np.arange(10)                                  # copies 0–9 of each class for training
test_idx = np.arange(500, 700)                           # copies 500–699 of each class for testing (never trained on)
few_x, few_y = noisy_bank[:, few_idx].reshape(-1, 32, 32), np.repeat(np.arange(4), len(few_idx))
test_x, test_y = noisy_bank[:, test_idx].reshape(-1, 32, 32), np.repeat(np.arange(4), len(test_idx))
test_loader_2d = DataLoader(ParticleDataset(test_x, test_y), batch_size=256)
aug_results = {}
for name, aug in [("no augmentation", None), ("interpolated rotation", augment_interpolated_rotation),
                  ("rot90 + flip (exact)", augment_rot90_flip)]:
    torch.manual_seed(SEED)
    model = SmallCNN2D()
    loader = DataLoader(ParticleDataset(few_x, few_y, augment=aug), batch_size=40, shuffle=True)
    hist = fit(model, loader, test_loader_2d, epochs=300, lr=1e-3, eval_every=25, verbose=False)
    aug_results[name] = hist
    print(f"{name:24s}: final train acc {hist['train_acc'][-1]:.2f} | test acc {hist['val_acc'][-1]:.2f} "
          f"| best test acc {max(hist['val_acc']):.2f} | {hist['seconds']:.0f} s")

fig, ax = plt.subplots(figsize=(6, 3.2))
for (name, h), col in zip(aug_results.items(), ["tab:gray", "tab:orange", "tab:green"]):
    ax.plot(h["epoch"], h["train_acc"], ls="--", color=col, lw=1)
    ax.plot(h["epoch"], h["val_acc"], ls="-", color=col, lw=2, label=name)
ax.axhline(0.25, color="k", lw=0.5)
ax.set_xlabel("epoch (40 images each)"); ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("10 labels per class — dashed: training, solid: unseen images"); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without augmentation the network reaches 100 % training accuracy quickly — it has
memorised 40 images — and stalls at a lower accuracy on new images: the gap between the dashed and solid lines *is*
overfitting. Both augmentations give every image many legitimate variants and generalise clearly better (0.77–0.81
instead of 0.65 here). Exact 90°/flip augmentation does so without changing a single pixel value. The interpolated
rotation covers *all* angles, but it also smooths each image and leaves empty corners — statistics that never occur at
test time. Here that did not hurt (it even scored best); with sharper images, other noise levels or other random seeds
(`torch.manual_seed(1)`, `2`) it may, so measure rather than assume.

💡 **Lesson for every later module:** augmentations must respect both the **geometry** (only physically possible poses)
and the **imaging statistics** (noise, missing wedge). DISCA (Module 22) duplicates each subtomogram with a random
rotation and, in the paper, fills the empty corners with Gaussian noise for exactly this reason. In 3D there is one more
trap: a single flip is a **mirror image** (it turns a left-handed helix right-handed) — not a rotation. Only proper
rotations (determinant +1) are valid poses; Module 02 showed that a careless "random rotation" can be a reflection.

(Strictly, here we used the 800 test images as the "validation" set to plot curves — fine for a demonstration, but in a
real study that would be choosing settings on the test set. Section 7 does it properly.)

---
## 7 · A 2D CNN that tells four molecules apart

Now the full recipe with a proper three-way split of the projection bank: copies 0–399 of each class for **training**
(1,600 images, with exact 90°/flip augmentation), 400–499 for **validation** (400), 500–699 for **testing** (800, the
same test images as Section 6). SNR 0.1 — a realistic single-particle or good-tomogram noise level.

In [ ]:
# ── 7.1 Train the 2D CNN ──
rng = np.random.default_rng(SEED + 7)
SNR_2D = 0.1
noisy_01 = np.stack([[standardize(add_noise_snr(bank[c, j], SNR_2D, rng)) for j in range(N_PER_CLASS)] for c in range(4)])


def take(arr: np.ndarray, idx) -> tuple[np.ndarray, np.ndarray]:
    """Images arr[class, idx] flattened to [n, 32, 32] with matching labels."""
    idx = np.asarray(idx)
    return arr[:, idx].reshape(-1, *arr.shape[2:]), np.repeat(np.arange(arr.shape[0]), len(idx))


tr_x, tr_y = take(noisy_01, range(0, 400))
va_x, va_y = take(noisy_01, range(400, 500))
te_x, te_y = take(noisy_01, range(500, 700))
train_loader = DataLoader(ParticleDataset(tr_x, tr_y, augment=augment_rot90_flip), batch_size=64, shuffle=True)
val_loader = DataLoader(ParticleDataset(va_x, va_y), batch_size=256)
test_loader = DataLoader(ParticleDataset(te_x, te_y), batch_size=256)
torch.manual_seed(SEED)
cnn2d = SmallCNN2D()
hist2d = fit(cnn2d, train_loader, val_loader, epochs=12, lr=2e-3)

In [ ]:
# ── 7.2 Learning curves ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
a1.plot(hist2d["epoch"], hist2d["train_loss"], "o-", label="training"); a1.plot(hist2d["epoch"], hist2d["val_loss"], "s-", label="validation")
a1.set_xlabel("epoch"); a1.set_ylabel("cross-entropy"); a1.legend(fontsize=7); a1.set_title("loss")
a2.plot(hist2d["epoch"], hist2d["train_acc"], "o-", label="training"); a2.plot(hist2d["epoch"], hist2d["val_acc"], "s-", label="validation")
a2.set_xlabel("epoch"); a2.set_ylabel("accuracy"); a2.legend(fontsize=7); a2.set_title("accuracy")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both curves improve together for the first epochs. When the validation loss stops
improving while the training loss keeps falling, the network has started to fit noise; `fit` kept the weights of the best
validation epoch. Only now do we look at the test set — once.

### 7.3 Confusion matrix and per-class precision/recall

A **confusion matrix** counts, for each true class (row), how many test particles were predicted as each class
(column). Precision and recall per class (Module 04) follow directly: recall = diagonal ÷ row sum, precision =
diagonal ÷ column sum.

In [ ]:
# ── 7.3 Test-set evaluation ──
test_loss, test_acc, pred, true = evaluate(cnn2d, test_loader)
cm = confusion_matrix(true, pred)
prec, rec, f1, _ = precision_recall_fscore_support(true, pred)
print(f"TEST accuracy {test_acc:.3f} on {len(true)} unseen particles (SNR {SNR_2D})")
for c in range(4):
    by_hand_rec = cm[c, c] / cm[c].sum()
    print(f"  {CLASS_NAMES[c]:18s} precision {prec[c]:.2f}  recall {rec[c]:.2f} (by hand {by_hand_rec:.2f})  F1 {f1[c]:.2f}")
assert test_acc > 0.6, "the 2D CNN should beat chance (0.25) by a wide margin"

fig, ax = plt.subplots(figsize=(4.4, 3.8))
ax.imshow(cm, cmap="viridis")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, v, ha="center", va="center", color="w" if v < cm.max() / 2 else "k", fontsize=9)
ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"2D CNN, test accuracy {test_acc:.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** RuBisCO — the smallest and most compact molecule — is recognised best (F1 ≈ 0.92). Most
errors involve the **capped proteasome**: it is mistaken for the 20S proteasome (from many directions the cap is a small
extra blob that noise easily hides) and for the ribosome (a capped barrel seen from the side is as long as a ribosome).
The confusion matrix shows *which* biology is hard, which a single accuracy number hides.

### 7.4 What did the first layer learn, and how robust is the network to noise?

In [ ]:
# ── 7.4 First-layer filters and accuracy vs test SNR ──
filters = cnn2d.features[0].weight.detach().cpu()[:, 0]          # 8 learned 3×3 filters
rng = np.random.default_rng(SEED + 74)
snr_tests = [1.0, 0.3, 0.1, 0.05, 0.02]
acc_vs_snr = []
for s in snr_tests:
    xs = np.stack([standardize(add_noise_snr(bank[c, j], s, rng)) for c in range(4) for j in range(500, 700)])
    acc_vs_snr.append(evaluate(cnn2d, DataLoader(ParticleDataset(xs, te_y), batch_size=256))[1])
print("test accuracy at SNR", dict(zip(snr_tests, np.round(acc_vs_snr, 3))), f"(trained at SNR {SNR_2D})")

mosaic = np.full((2 * 4 - 1, 4 * 4 - 1), np.nan)          # 2 rows × 4 columns of 3×3 filters, 1-pixel gaps
for i in range(8):
    r, c = divmod(i, 4)
    mosaic[4 * r:4 * r + 3, 4 * c:4 * c + 3] = filters[i] / filters[i].abs().max()
fig, (a0, ax) = plt.subplots(1, 2, figsize=(9.5, 2.9))
im = a0.imshow(mosaic, cmap="RdBu_r", vmin=-1, vmax=1); a0.axis("off")
a0.set_title("the 8 learned 3×3 first-layer filters", fontsize=9); plt.colorbar(im, ax=a0, fraction=0.03, label="weight (scaled)")
ax.plot(snr_tests, acc_vs_snr, "o-"); ax.axvline(SNR_2D, color="gray", ls=":", label="training SNR")
ax.set_xscale("log"); ax.set_xlabel("SNR of test images"); ax.set_ylabel("accuracy"); ax.set_ylim(0, 1)
ax.axhline(0.25, color="k", lw=0.5); ax.legend(fontsize=7); ax.set_title("accuracy vs noise")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The first-layer filters (red = positive weight, blue = negative) are small smoothing and
edge-like detectors — at SNR 0.1 a sensible first step is to average neighbouring pixels. Accuracy falls steeply below
the training SNR — and, perhaps surprisingly, it improves only modestly on *cleaner* images (≈ 0.78 → 0.87 at ten
times the SNR), which are also unlike anything it was trained on: a network is only as good as (and only as general
as) the data it was trained on. When real data
differ from the training data (different noise, CTF, crowding), performance drops — the **domain shift** problem of
Module 20.

---
## 8 · A tiny 3D CNN on subtomograms with a missing wedge

Cryo-ET gives **3D** subtomograms, and every convolution, pooling and tensor simply gains one dimension:
`nn.Conv3d`, `nn.MaxPool3d`, tensors of shape `[batch, 1, z, y, x]`. We simulate subtomograms as the Xu-lab benchmarks
do (Module 07–09): random rotation → **missing wedge** (tilt range ±60°, so Fourier coefficients with
$|k_z| > |k_x|\tan 60°$ are lost) → Gaussian noise at **SNR 0.05**. We first resample the 32³ maps (10 Å voxels) to
**24³ boxes at 13.3 Å per voxel** — close to the 13.33 Å binning of DISCA's subtomograms.

**Augmentation in 3D.** A rotated subtomogram also has a rotated missing wedge, so arbitrary rotations would create
wedge directions that never occur in the data. The 180° rotations about the x, y and z axes, however, map the wedge
onto itself and move voxels exactly (each equals flipping *two* axes). These four proper rotations are our augmentation.

**Batch normalisation.** 3D networks on very noisy inputs often sit on a "plateau" at chance accuracy for many epochs
before they start learning (try removing the `BatchNorm3d` layers below to see it). A **batch-normalisation** layer
rescales each channel's outputs to zero mean and unit variance over the current mini-batch (then applies a learned scale
and shift), which keeps signals well-sized through the network and makes training faster and more reliable. During
evaluation it uses running averages collected in training — one reason we call `model.eval()` before testing.

In [ ]:
# ── 8.1 Simulate 24³ subtomograms ──
rng = np.random.default_rng(SEED + 8)
templates24 = {pid: np.clip(ndimage.zoom(v, 0.75, order=1), 0, None) for pid, v in zoo.items()}  # 24³ at 13.3 Å
N_SUB = 250                                                       # subtomograms per class
t0 = time.time()
sub_x, sub_y = [], []
for c, pid in enumerate(CAST):
    for _ in range(N_SUB):
        v, _, _ = simulate_subtomogram(templates24[pid], rng, snr=0.05, tilt_max_deg=60)
        sub_x.append(normalize(v)); sub_y.append(c)
sub_x, sub_y = np.array(sub_x), np.array(sub_y)
print(f"{len(sub_y)} subtomograms of shape {sub_x.shape[1:]} in {time.time() - t0:.1f} s")

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for c in range(4):
    i = np.where(sub_y == c)[0][0]
    axes[0, c].imshow(sub_x[i][12]); axes[0, c].set_title(f"{CLASS_NAMES[c]}\nxy slice", fontsize=8)
    axes[1, c].imshow(sub_x[i][:, 12]); axes[1, c].set_title("xz slice (wedge smears along z)", fontsize=7)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# ── 8.2 Wedge-preserving augmentation and the 3D network ──
def augment_rot180_3d(x: torch.Tensor) -> torch.Tensor:
    """Random proper 180° rotation about x, y or z (or none) of a [1, z, y, x] volume; preserves a y-tilt wedge."""
    k = int(torch.randint(4, (1,)))
    pairs = [None, (-2, -1), (-3, -1), (-3, -2)]   # none, about z (flip y,x), about y (flip z,x), about x (flip z,y)
    return x if pairs[k] is None else x.flip(pairs[k])


R_about_y = np.diag([-1.0, 1.0, -1.0])                 # flipping z and x = rotating 180° about y
print("det of the 'flip z and x' matrix:", np.linalg.det(R_about_y), "→ a proper rotation;",
      "flipping one axis has det", np.linalg.det(np.diag([-1.0, 1.0, 1.0])), "→ a mirror image")
mask = missing_wedge_mask((24, 24, 24), 60.0)          # Fourier-space wedge (unshifted FFT layout)
neg = (-np.arange(24)) % 24                              # index of the frequency −k
for axes_flipped in [(1, 2), (0, 2), (0, 1)]:            # the three 180° rotations negate two frequency axes
    m = mask
    for ax_ in axes_flipped:
        m = np.take(m, neg, axis=ax_)
    assert np.array_equal(m, mask), "each 180° rotation must map the missing wedge onto itself"
print("✅ the ±60° missing wedge is unchanged by all three 180° rotations")


class TinyCNN3D(nn.Module):
    """[conv3d(8) → BN → ReLU → maxpool] → [conv3d(16) → BN → ReLU → maxpool] → flatten → FC(64) → ReLU → FC(n)."""

    def __init__(self, n_classes: int = 4, size: int = 24):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1), nn.BatchNorm3d(8), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(8, 16, kernel_size=3, padding=1), nn.BatchNorm3d(16), nn.ReLU(), nn.MaxPool3d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * (size // 4) ** 3, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


print("TinyCNN3D parameters:", sum(p.numel() for p in TinyCNN3D().parameters()))

In [ ]:
# ── 8.3 Train and test the 3D CNN ──
idx_by_class = [np.where(sub_y == c)[0] for c in range(4)]
tr3 = np.concatenate([ix[:150] for ix in idx_by_class])          # 150 / 30 / 70 per class
va3 = np.concatenate([ix[150:180] for ix in idx_by_class])
te3 = np.concatenate([ix[180:] for ix in idx_by_class])
tr3_loader = DataLoader(ParticleDataset(sub_x[tr3], sub_y[tr3], augment=augment_rot180_3d), batch_size=32, shuffle=True)
va3_loader = DataLoader(ParticleDataset(sub_x[va3], sub_y[va3]), batch_size=64)
te3_loader = DataLoader(ParticleDataset(sub_x[te3], sub_y[te3]), batch_size=64)
torch.manual_seed(SEED)
cnn3d = TinyCNN3D()
hist3d = fit(cnn3d, tr3_loader, va3_loader, epochs=12, lr=1e-3)
_, acc3, pred3, true3 = evaluate(cnn3d, te3_loader)
cm3 = confusion_matrix(true3, pred3)
print(f"3D CNN TEST accuracy: {acc3:.3f} on {len(true3)} unseen subtomograms (SNR 0.05, ±60° wedge)")
assert acc3 > 0.5, "the 3D CNN should clearly beat chance (0.25); if not, re-run the cell (unlucky initialisation)"

# a different microscope setting at test time: a larger missing wedge (±40°)
rng = np.random.default_rng(SEED + 83)
x40 = np.array([normalize(simulate_subtomogram(templates24[pid], rng, snr=0.05, tilt_max_deg=40)[0])
                for pid in CAST for _ in range(70)])
acc40 = evaluate(cnn3d, DataLoader(ParticleDataset(x40, np.repeat(np.arange(4), 70)), batch_size=64))[1]
print(f"same network on ±40° subtomograms (never seen in training): accuracy {acc40:.3f}")

fig, ax = plt.subplots(figsize=(4.4, 3.8))
ax.imshow(cm3, cmap="viridis")
for (i, j), v in np.ndenumerate(cm3):
    ax.text(j, i, v, ha="center", va="center", color="w" if v < cm3.max() / 2 else "k", fontsize=9)
ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"3D CNN, test accuracy {acc3:.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With only 600 training subtomograms and 12 epochs, a network with ~225,000 parameters
separates the four molecules from 3D data that are too noisy to read by eye — 3D gives it the full shape, not one
projection. The capped/uncapped proteasome pair again carries most of the errors. A larger missing wedge at test time,
a setting the network never saw, typically costs accuracy: another face of domain shift.

✏️ **Your turn.** (a) Lower the SNR in cell 8.1 to 0.02 and re-run 8.1–8.3. (b) Remove the augmentation
(`augment=None`) in 8.3. (c) Train with `tilt_max_deg=40` data instead and test on ±60°.

<details><summary>What to expect</summary>Lower SNR lowers accuracy, mostly through more proteasome confusions; without augmentation the network overfits sooner (watch the training vs validation accuracy); training on the harder (±40°) wedge usually transfers better to ±60° than the reverse.</details>

---
## 9 · GPUs, benchmarks and what comes next

**GPU note.** Everything above runs on a CPU because the networks are tiny. On Colab, *Runtime → Change runtime type →
T4 GPU* makes `DEVICE` become `cuda`; the code already moves models and batches with `.to(DEVICE)`. Real subtomogram
networks (32³–64³ inputs, millions of parameters, 10⁵–10⁶ particles) need a GPU.

### 📄 Paper connection — SHREC'19 and SHREC'20: community benchmarks for cryo-ET classification

**Papers.** Gubins I, van der Schot G, Veltkamp RC, Förster F, Du X, Zeng X, Zhu Z, Chang L, **Xu M**, *et al.* (2019).
*Classification in cryo-electron tomograms (SHREC'19 track).* Eurographics Workshop on 3D Object Retrieval.
[doi:10.2312/3dor.20191061](https://doi.org/10.2312/3dor.20191061). — Gubins I, Chaillet ML, van der Schot G, …, Zeng X,
**Xu M**, *et al.* (2020). *SHREC 2020: Classification in cryo-electron tomograms.* **Computers & Graphics** 91:279–289.
[doi:10.1016/j.cag.2020.07.010](https://doi.org/10.1016/j.cag.2020.07.010) (arXiv:2203.10035).

- **Question it answers.** How well do different methods *localise and classify* proteins in tomograms, judged on the
  same public, fully annotated data?
- **What the benchmark does.** Simulated cell-like tomograms with 12 protein classes (SHREC'20: from 42.75 kDa, PDB 1S3X —
  our Hsp70 domain — to 3,325.59 kDa, PDB 5MRC), plus vesicles and gold fiducials; 300 kV, ±60°, defocus 2–5 µm. In
  SHREC'20, **tomograms 0–7 are for training/validation and tomogram 8 for testing** — a tomogram-level split. Methods are
  scored by precision, recall, F1, miss rate and centre distance.
- **What our toy keeps.** Simulated ground truth, a held-out *tomogram* for honest testing, per-class precision/recall,
  and the observation that small molecules are hardest.
- **What our toy leaves out.** Localisation in full tomograms (we classify pre-cut boxes), 12 classes, realistic
  crowding and CTF.
- **Result reported (SHREC'20).** The authors report that every learning-based method beat template matching
  (F1 0.372 for TM); DeepFinder had the best overall F1 (0.868); performance correlates strongly with molecular weight
  (e.g. one method scored F1 0.594 on proteins < 200 kDa and 0.989 on proteins > 600 kDa). Which entry was the Xu Lab's is
  not stated in our notes.
- **Code / data.** Benchmark data are distributed by the organisers; see the papers.

### 📄 Preview — the lab's first deep subtomogram classifier (Module 16)

**Xu M, Chai X, Muthakana H, Liang X, Yang G, Zeev-Ben-Mordehai T, Xing E (2017).** *Deep learning based subdivision
approach for large scale macromolecules structure recovery from electron cryo tomograms.* **Bioinformatics**
33(14):i13–i22 (ISMB/ECCB 2017). [doi:10.1093/bioinformatics/btx230](https://doi.org/10.1093/bioinformatics/btx230)
- Two 3D CNNs (Inception3D and the VGG-like **DSRF3D**: small 3×3×3 kernels, max-pooling, fully connected layers — the
  same plan as our `TinyCNN3D`) trained with cross-entropy on simulated 40³ subtomograms of 22 complexes + an empty class.
- Key idea beyond classification: the **pre-softmax outputs** serve as features to *subdivide* millions of particles into
  small, more homogeneous groups (k-means, k = 100) before template-free averaging.
- The authors report, e.g., at SNR 0.03 and ±60°: Inception3D 0.608, DSRF3D 0.658 vs 0.446 for a rotation-invariant-feature
  SVM. Code: AITom `aitom/classify/deep/supervised/cnn/subdivide.py`.

**Where this leads.** Module 16 (3D CNN classification, imbalance, distillation), 17 (interpretability and
segmentation), 19 (few labels), 20 (simulation → reality), 21–23 (unsupervised alignment, clustering, disentanglement),
22–24 (denoising, picking, segmentation with foundation models).

---
## ✅ Checkpoint

1. In one sentence: what does `loss.backward()` do, and what does `optimizer.step()` do?
   <details><summary>Answer</summary><code>backward()</code> computes the gradient of the loss with respect to every parameter (by automatic differentiation); <code>step()</code> moves each parameter a small amount against its gradient (downhill), scaled by the learning rate.</details>
2. Why does an MLP need nonlinear activation functions?
   <details><summary>Answer</summary>Without them, a stack of linear layers is just one linear function, which can only draw straight (flat) decision boundaries — it could not separate inside from outside a vesicle.</details>
3. A paper reports 98 % accuracy, splitting 10,000 subtomograms from 5 tomograms randomly 80/20. What is your first question?
   <details><summary>Answer</summary>Were test particles from the same tomograms as training particles? If so the number may be inflated by tomogram-specific shortcuts (data leakage); ask for a tomogram-level (or cell-level) split.</details>
4. Arbitrary-angle rotation augmentation can hurt (Section 6). Why, and how could you fix it?
   <details><summary>Answer</summary>Interpolation smooths the noise and leaves empty corners, so training images have statistics unlike test images (in our run it still helped — measure rather than assume). Fixes: use exact symmetries (90° rotations, flips), apply a circular mask to all images, fill corners with matching noise, or rotate clean data before adding noise in simulation.</details>
5. Which 3D "augmentations" are forbidden for subtomograms, and why?
   <details><summary>Answer</summary>Single-axis flips (mirror images, determinant −1) — molecules are chiral, so a mirror image is not a possible pose; and arbitrary rotations if the missing wedge is not rotated consistently (they create wedge orientations absent from real data).</details>
6. In the confusion matrices, which pair of molecules was most often confused, and why is that biologically interesting?
   <details><summary>Answer</summary>20S vs singly capped proteasome: they differ only by the regulatory cap — a compositional/conformational variant. Distinguishing such states is a central goal of in-situ structural biology (Modules 14, 22, 23).</details>

## 🧠 Questions & discussion

1. The leaky split in Section 5 gave a higher number. Describe a realistic cryo-ET project in which a *cell-level* or
   *grid-level* split would be needed instead of a tomogram-level split.
2. Our networks were trained on simulated data with a known SNR. List three ways real subtomograms differ from our
   simulations, and predict how each would affect accuracy (Module 20 tests this).
3. A network reaches 99 % accuracy on ribosome vs RuBisCO. Is that a scientific result? What control experiment would
   convince you the network uses *shape* rather than *size* or *total density*?
4. Supervised networks need labels. Where would the labels come from for a cell type nobody has imaged before?
   (Modules 19, 22 and 31 give three different answers.)
5. Why might the validation accuracy fluctuate from epoch to epoch even when training is going well?

## 🚀 Bonus — going further

- **Batch normalisation and dropout.** Add `nn.BatchNorm2d` after each convolution and `nn.Dropout(0.5)` in the head of
  `SmallCNN2D`; compare learning curves.
- **Global pooling.** Replace `Flatten` by `nn.AdaptiveMaxPool2d(1)` (global max pooling, the core of DISCA's YOPO
  feature extractor, Module 22). What happens to translation sensitivity? Test by shifting test images by 3 pixels.
- **Learning-rate schedules.** Add `torch.optim.lr_scheduler.StepLR` to `fit` and see whether the 3D CNN converges faster.
- **Deep dives in this repository.** [`../../../../sci_img/tutorial/ComputationalSignalBase/PyTorchTensorBasic/`](../../../../sci_img/tutorial/ComputationalSignalBase/PyTorchTensorBasic/)
  (tensors in depth) and [`../../SubtomogramClassification/`](../../SubtomogramClassification/) (a longer subtomogram
  classification tutorial).
- **Lab code.** Read `aitom/classify/deep/supervised/cnn/` in [AITom](https://github.com/xulabs/aitom) and identify the
  DSRF3D layers.

## 📝 Lab note

```text
Module 05 — date:
Input: zoo projections (32×32, 1 nm) and 24³ subtomograms (13.3 Å, ±60° wedge), SNR 0.05–0.2
Output: trained MLP / 2D CNN / 3D CNN; test accuracies; confusion matrices
Method: autograd + Adam; cross-entropy; Dataset/DataLoader; early stopping; batch norm; augmentation (rot90/flip; 180° 3D)
Parameter I changed and what happened:
A failure mode / surprise (leaky split, interpolation augmentation, ±40° wedge):
Paper link: SHREC'19/'20 (Gubins et al.); Xu et al. 2017 (Module 16); DISCA augmentation (Module 22)
```

## 📚 References

1. Scott M, Gunderson CW, Mateescu EM, Zhang Z, Hwa T. Interdependence of cell growth and gene expression: origins and consequences. *Science* 330:1099–1102 (2010). doi:10.1126/science.1192588
2. LeCun Y, Bengio Y, Hinton G. Deep learning. *Nature* 521:436–444 (2015). doi:10.1038/nature14539
3. Kingma DP, Ba J. Adam: a method for stochastic optimization. arXiv:1412.6980 (2014).
4. Paszke A *et al.* PyTorch: an imperative style, high-performance deep learning library. *NeurIPS* 32 (2019). arXiv:1912.01703
5. Kaufman S, Rosset S, Perlich C, Stitelman O. Leakage in data mining: formulation, detection, and avoidance. *ACM Trans Knowl Discov Data* 6:15 (2012). doi:10.1145/2382577.2382579
6. Gubins I *et al.* Classification in cryo-electron tomograms. SHREC'19, Eurographics 3DOR (2019). doi:10.2312/3dor.20191061
7. Gubins I *et al.* SHREC 2020: Classification in cryo-electron tomograms. *Computers & Graphics* 91:279–289 (2020). doi:10.1016/j.cag.2020.07.010
8. Xu M, Chai X, Muthakana H, Liang X, Yang G, Zeev-Ben-Mordehai T, Xing E. Deep learning based subdivision approach for large scale macromolecules structure recovery from electron cryo tomograms. *Bioinformatics* 33:i13–i22 (2017). doi:10.1093/bioinformatics/btx230
9. Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M. High-throughput cryo-ET structural pattern mining by unsupervised deep iterative subtomogram clustering. *PNAS* 120:e2213149120 (2023). doi:10.1073/pnas.2213149120

## ➡️ Next

**[06 · How an electron microscope makes an image: projection, dose and the CTF](06_electron_image_formation.ipynb)** —
Part B begins: where the noise, the "rings" and the contrast of real images come from. Course overview:
[`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).